# Product-Qualified Account Scoring

## Senior Data Science Task

Build a leakage-aware machine learning system that predicts which accounts are most likely to become **Won / Closed Won** opportunities in the next 90 days.

This notebook uses the SaaS seed data in:

`lightdash-demo-saas/seeds`

The task is account-level prioritization for Sales, Growth, and Product teams.

### Business Question

Which accounts should the team prioritize this week because their product usage, marketing source, sales activity, firmographics, and geography suggest high conversion potential?

### Prediction Target

For each account snapshot date:

```text
Target = 1 if the account creates a Won / Closed Won deal in the next 90 days
Target = 0 otherwise
```

### Senior-Level Requirements Covered

- Time-based train/validation/test split
- Leakage prevention with snapshot dates
- Product usage features from event telemetry
- Sales and marketing activity features
- Firmographic and geography features
- Multiple models
- Probability calibration using a validation period
- Per-model evaluation metrics
- Precision@K, Recall@K, lift, and revenue capture
- Calibration tables
- Ranked account scoring output

### Important Data Limitation

`deals_raw.csv` has `created_date` and current/final `stage`, but no explicit close date. For this exercise, we use `created_date` as the opportunity date. In production, replace this with true opportunity close date or stage transition timestamp.

In [ ]:
# If needed, install dependencies in your notebook environment:
# %pip install pandas numpy scikit-learn matplotlib seaborn

from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SEED_DIR = Path("lightdash-demo-saas/seeds")
PREDICTION_HORIZON_DAYS = 90
OBSERVATION_WINDOWS = [7, 14, 30, 60, 90]
HIGH_INTENT_EVENTS = [
    "workspace_created",
    "report_generated",
    "dashboard_viewed",
    "workflow_created",
    "api_call_made",
    "report_exported",
]
WON_STAGES = {"won", "closed won"}

## 1. Load Data

We load all available seed tables, then parse dates and normalize IDs. The model will mainly use account, user, product event, deal, marketing lead, sales activity, and geography tables.

In [ ]:
def read_csv(name: str, date_cols: list[str] | None = None) -> pd.DataFrame:
    path = SEED_DIR / name
    df = pd.read_csv(path)
    for col in date_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

accounts = read_csv("accounts_raw.csv")
users = read_csv(
    "users_raw.csv",
    ["created_at", "first_logged_in_at", "latest_logged_in_at"],
)
tracks = read_csv("tracks_raw.csv", ["event_timestamp"])
deals = read_csv("deals_raw.csv", ["created_date"])
user_deals = read_csv("user_deals_raw.csv")
marketing_leads = read_csv("marketing_leads.csv", ["created_at", "converted_at"])
activities = read_csv("activities_raw.csv", ["activity_timestamp"])
lead_geo = read_csv("lead_geographic_data.csv")
addresses = read_csv("addresses_raw.csv", ["valid_from", "valid_to"])

print("accounts", accounts.shape)
print("users", users.shape)
print("tracks", tracks.shape)
print("deals", deals.shape)
print("marketing_leads", marketing_leads.shape)
print("activities", activities.shape)

In [ ]:
# Basic cleanup
accounts["estimated_annual_recurring_revenue"] = pd.to_numeric(
    accounts["estimated_annual_recurring_revenue"], errors="coerce"
)
users["experience_in_years"] = pd.to_numeric(users["experience_in_years"], errors="coerce")
users["is_marketing_opted_in"] = pd.to_numeric(users["is_marketing_opted_in"], errors="coerce").fillna(0)
deals["amount"] = pd.to_numeric(deals["amount"], errors="coerce").fillna(0)
deals["seats"] = pd.to_numeric(deals["seats"], errors="coerce")
marketing_leads["lead_cost"] = pd.to_numeric(marketing_leads["lead_cost"], errors="coerce").fillna(0)
activities["call_duration_seconds"] = pd.to_numeric(activities["call_duration_seconds"], errors="coerce")

deals["is_won"] = deals["stage"].str.lower().isin(WON_STAGES).astype(int)

# User/account mapping for product usage aggregation
tracks = tracks.merge(users[["user_id", "account_id", "first_logged_in_at"]], on="user_id", how="left")
marketing_leads = marketing_leads.merge(
    users[["user_id", "account_id"]], on="user_id", how="left", suffixes=("", "_from_user")
)
marketing_leads["account_id"] = marketing_leads["account_id"].fillna(marketing_leads.get("account_id_from_user"))
activities = activities.merge(
    marketing_leads[["lead_id", "user_id", "account_id"]], on="lead_id", how="left"
)

print("Won stage rate in deals:", deals["is_won"].mean().round(3))
print("Top product events:")
print(tracks["event_name"].value_counts().head(12))

## 2. Define Leakage-Aware Snapshot Dates

A row in the modeling dataset is an `account_id` at a `snapshot_date`.

Features only use information available **on or before** the snapshot date.

The target looks forward 90 days from that snapshot date.

We exclude accounts that already had a won deal before the snapshot because the business question is prioritizing accounts that have not yet converted.

In [ ]:
def make_snapshot_dates() -> pd.DatetimeIndex:
    min_date = max(
        users["created_at"].min(),
        tracks["event_timestamp"].min(),
        marketing_leads["created_at"].min(),
    )
    max_date = deals["created_date"].max() - pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

    # Monthly snapshots keep the dataset compact and enforce temporal evaluation.
    dates = pd.date_range(
        min_date.normalize() + pd.offsets.MonthBegin(1),
        max_date.normalize(),
        freq="MS",
    )
    return dates

snapshot_dates = make_snapshot_dates()
snapshot_dates

## 3. Feature Engineering

The feature set combines:

- Firmographics from `accounts_raw.csv`
- User base profile from `users_raw.csv`
- Product usage from `tracks_raw.csv`
- Marketing acquisition from `marketing_leads.csv`
- Sales activity from `activities_raw.csv`
- Pipeline context from `deals_raw.csv`
- Geography from `lead_geographic_data.csv`

This is intentionally account-level because Sales/Growth prioritization happens at the account level.

In [ ]:
def safe_divide(numerator: float, denominator: float) -> float:
    if denominator in (0, 0.0) or pd.isna(denominator):
        return 0.0
    return numerator / denominator


def mode_or_unknown(series: pd.Series) -> str:
    series = series.dropna()
    if series.empty:
        return "Unknown"
    return str(series.mode().iloc[0])


def days_since(snapshot_date: pd.Timestamp, value: pd.Timestamp | None) -> float:
    if value is None or pd.isna(value):
        return np.nan
    return float((snapshot_date - value).days)


def build_features_for_snapshot(snapshot_date: pd.Timestamp) -> pd.DataFrame:
    horizon_end = snapshot_date + pd.Timedelta(days=PREDICTION_HORIZON_DAYS)

    base = accounts.copy()
    base["snapshot_date"] = snapshot_date

    # -------------------------
    # Account/user features
    # -------------------------
    users_seen = users[users["created_at"] <= snapshot_date].copy()
    user_features = users_seen.groupby("account_id").agg(
        users_created=("user_id", "nunique"),
        users_first_logged_in=("first_logged_in_at", lambda s: s.le(snapshot_date).sum()),
        marketing_opt_in_rate=("is_marketing_opted_in", "mean"),
        avg_experience_years=("experience_in_years", "mean"),
        max_experience_years=("experience_in_years", "max"),
        primary_job_title=("job_title", mode_or_unknown),
    ).reset_index()
    base = base.merge(user_features, on="account_id", how="left")

    # -------------------------
    # Product usage features
    # -------------------------
    tracks_seen = tracks[
        (tracks["event_timestamp"] <= snapshot_date)
        & tracks["account_id"].notna()
    ].copy()

    if not tracks_seen.empty:
        usage_all = tracks_seen.groupby("account_id").agg(
            total_events_to_date=("event_id", "count"),
            active_users_to_date=("user_id", "nunique"),
            distinct_events_to_date=("event_name", "nunique"),
            last_event_at=("event_timestamp", "max"),
            first_event_at=("event_timestamp", "min"),
        ).reset_index()
        usage_all["days_since_last_event"] = usage_all["last_event_at"].apply(lambda x: days_since(snapshot_date, x))
        usage_all["days_since_first_event"] = usage_all["first_event_at"].apply(lambda x: days_since(snapshot_date, x))
        usage_all = usage_all.drop(columns=["last_event_at", "first_event_at"])
        base = base.merge(usage_all, on="account_id", how="left")

        for event_name in HIGH_INTENT_EVENTS:
            event_counts = (
                tracks_seen[tracks_seen["event_name"] == event_name]
                .groupby("account_id")
                .size()
                .rename(f"event_count__{event_name}")
                .reset_index()
            )
            base = base.merge(event_counts, on="account_id", how="left")

    for window in OBSERVATION_WINDOWS:
        start = snapshot_date - pd.Timedelta(days=window)
        window_tracks = tracks[
            (tracks["event_timestamp"] > start)
            & (tracks["event_timestamp"] <= snapshot_date)
            & tracks["account_id"].notna()
        ]
        window_features = window_tracks.groupby("account_id").agg(
            **{
                f"events_last_{window}d": ("event_id", "count"),
                f"active_users_last_{window}d": ("user_id", "nunique"),
                f"distinct_events_last_{window}d": ("event_name", "nunique"),
            }
        ).reset_index()
        base = base.merge(window_features, on="account_id", how="left")

    # Recent usage momentum: last 30 days vs previous 30 days.
    last_30 = tracks[
        (tracks["event_timestamp"] > snapshot_date - pd.Timedelta(days=30))
        & (tracks["event_timestamp"] <= snapshot_date)
        & tracks["account_id"].notna()
    ].groupby("account_id").size().rename("events_last_30d_raw")
    prev_30 = tracks[
        (tracks["event_timestamp"] > snapshot_date - pd.Timedelta(days=60))
        & (tracks["event_timestamp"] <= snapshot_date - pd.Timedelta(days=30))
        & tracks["account_id"].notna()
    ].groupby("account_id").size().rename("events_prev_30d_raw")
    momentum = pd.concat([last_30, prev_30], axis=1).fillna(0).reset_index()
    momentum["event_momentum_30d"] = momentum.apply(
        lambda row: safe_divide(row["events_last_30d_raw"] - row["events_prev_30d_raw"], row["events_prev_30d_raw"] + 1),
        axis=1,
    )
    base = base.merge(momentum[["account_id", "event_momentum_30d"]], on="account_id", how="left")

    # -------------------------
    # Marketing lead features
    # -------------------------
    leads_seen = marketing_leads[
        (marketing_leads["created_at"] <= snapshot_date)
        & marketing_leads["account_id"].notna()
    ].copy()
    if not leads_seen.empty:
        lead_features = leads_seen.groupby("account_id").agg(
            leads_to_date=("lead_id", "nunique"),
            total_lead_cost=("lead_cost", "sum"),
            avg_lead_cost=("lead_cost", "mean"),
            converted_leads_to_date=("converted_at", lambda s: s.notna().sum()),
            primary_lead_source=("lead_source", mode_or_unknown),
            primary_campaign=("campaign_name", mode_or_unknown),
            primary_utm_medium=("utm_medium", mode_or_unknown),
            primary_sdr=("SDR", mode_or_unknown),
        ).reset_index()
        base = base.merge(lead_features, on="account_id", how="left")

        for status in ["new", "contacted", "qualified", "converted", "disqualified"]:
            status_counts = (
                leads_seen[leads_seen["lead_status"].eq(status)]
                .groupby("account_id")
                .size()
                .rename(f"lead_status_count__{status}")
                .reset_index()
            )
            base = base.merge(status_counts, on="account_id", how="left")

    # -------------------------
    # Sales activity features
    # -------------------------
    activities_seen = activities[
        (activities["activity_timestamp"] <= snapshot_date)
        & activities["account_id"].notna()
    ].copy()
    if not activities_seen.empty:
        activity_features = activities_seen.groupby("account_id").agg(
            sales_activities_to_date=("activity_id", "count"),
            active_leads_touched=("lead_id", "nunique"),
            avg_call_duration_seconds=("call_duration_seconds", "mean"),
            last_activity_at=("activity_timestamp", "max"),
        ).reset_index()
        activity_features["days_since_last_sales_activity"] = activity_features["last_activity_at"].apply(
            lambda x: days_since(snapshot_date, x)
        )
        activity_features = activity_features.drop(columns=["last_activity_at"])
        base = base.merge(activity_features, on="account_id", how="left")

        for activity_type in activities_seen["activity_type"].dropna().unique():
            type_counts = (
                activities_seen[activities_seen["activity_type"].eq(activity_type)]
                .groupby("account_id")
                .size()
                .rename(f"activity_count__{activity_type}")
                .reset_index()
            )
            base = base.merge(type_counts, on="account_id", how="left")

    # -------------------------
    # Prior pipeline features
    # Avoid using final stage as a feature because stage may be updated after snapshot.
    # -------------------------
    prior_deals = deals[deals["created_date"] <= snapshot_date].copy()
    if not prior_deals.empty:
        prior_deal_features = prior_deals.groupby("account_id").agg(
            prior_deals_to_date=("deal_id", "nunique"),
            prior_pipeline_amount=("amount", "sum"),
            prior_avg_deal_amount=("amount", "mean"),
            prior_max_seats=("seats", "max"),
            primary_plan_to_date=("plan", mode_or_unknown),
        ).reset_index()
        base = base.merge(prior_deal_features, on="account_id", how="left")

    # -------------------------
    # Geography features from lead geography.
    # -------------------------
    lead_geo_seen = lead_geo.merge(
        marketing_leads[["lead_id", "account_id", "created_at"]], on="lead_id", how="left"
    )
    lead_geo_seen = lead_geo_seen[
        (lead_geo_seen["created_at"] <= snapshot_date)
        & lead_geo_seen["account_id"].notna()
    ]
    if not lead_geo_seen.empty:
        geo_features = lead_geo_seen.groupby("account_id").agg(
            primary_continent=("continent", mode_or_unknown),
            primary_country=("country_name", mode_or_unknown),
            distinct_countries=("country_code", "nunique"),
        ).reset_index()
        base = base.merge(geo_features, on="account_id", how="left")

    # -------------------------
    # Target construction
    # -------------------------
    already_won = deals[
        (deals["created_date"] <= snapshot_date)
        & deals["is_won"].eq(1)
    ].groupby("account_id").size().rename("won_before_snapshot").reset_index()

    future_won = deals[
        (deals["created_date"] > snapshot_date)
        & (deals["created_date"] <= horizon_end)
        & deals["is_won"].eq(1)
    ].groupby("account_id").agg(
        target_won_90d=("deal_id", "nunique"),
        target_revenue_90d=("amount", "sum"),
    ).reset_index()

    base = base.merge(already_won, on="account_id", how="left")
    base = base.merge(future_won, on="account_id", how="left")
    base["won_before_snapshot"] = base["won_before_snapshot"].fillna(0)
    base["target_won_90d"] = base["target_won_90d"].fillna(0)
    base["target_revenue_90d"] = base["target_revenue_90d"].fillna(0)
    base["target"] = (base["target_won_90d"] > 0).astype(int)

    # Exclude accounts already won before the snapshot. We are predicting net-new conversion.
    base = base[base["won_before_snapshot"].eq(0)].copy()

    return base

In [ ]:
feature_frames = [build_features_for_snapshot(snapshot_date) for snapshot_date in snapshot_dates]
model_df = pd.concat(feature_frames, ignore_index=True)

# Fill count-like nulls with 0. Leave categorical nulls for imputation.
count_like_prefixes = (
    "users_", "event_", "events_", "active_", "distinct_", "lead_", "activity_", "sales_", "prior_"
)
for col in model_df.columns:
    if col.startswith(count_like_prefixes) or col in [
        "total_events_to_date",
        "active_users_to_date",
        "distinct_events_to_date",
        "total_lead_cost",
        "avg_lead_cost",
        "converted_leads_to_date",
        "event_momentum_30d",
        "days_since_last_event",
        "days_since_first_event",
        "days_since_last_sales_activity",
        "target_revenue_90d",
    ]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce").fillna(0)

print(model_df.shape)
print(model_df[["snapshot_date", "account_id", "target", "target_revenue_90d"]].head())
print("Target rate:", model_df["target"].mean().round(4))
print("Positive rows:", int(model_df["target"].sum()))

## 4. Time-Based Train / Validation / Test Split

We split by `snapshot_date`, not randomly. This is critical because account behavior and conversion patterns drift over time.

- Train: earliest 60% of snapshot dates
- Validation: next 20%, used for probability calibration and model selection
- Test: final 20%, used only once for final reporting

In [ ]:
unique_dates = np.array(sorted(model_df["snapshot_date"].unique()))
train_end = unique_dates[int(len(unique_dates) * 0.60)]
valid_end = unique_dates[int(len(unique_dates) * 0.80)]

train_mask = model_df["snapshot_date"] < train_end
valid_mask = (model_df["snapshot_date"] >= train_end) & (model_df["snapshot_date"] < valid_end)
test_mask = model_df["snapshot_date"] >= valid_end

train_df = model_df[train_mask].copy()
valid_df = model_df[valid_mask].copy()
test_df = model_df[test_mask].copy()

print("train", train_df.shape, train_df["snapshot_date"].min(), train_df["snapshot_date"].max(), train_df["target"].mean())
print("valid", valid_df.shape, valid_df["snapshot_date"].min(), valid_df["snapshot_date"].max(), valid_df["target"].mean())
print("test ", test_df.shape, test_df["snapshot_date"].min(), test_df["snapshot_date"].max(), test_df["target"].mean())

## 5. Preprocessing

We use numeric imputation/scaling and categorical one-hot encoding. The preprocessing is fit on train only.

In [ ]:
TARGET_COL = "target"
REVENUE_COL = "target_revenue_90d"
DROP_COLS = [
    "account_id",
    "account_name",
    "snapshot_date",
    "target",
    "target_won_90d",
    "target_revenue_90d",
    "won_before_snapshot",
]

feature_cols = [c for c in model_df.columns if c not in DROP_COLS]
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]
X_valid = valid_df[feature_cols]
y_valid = valid_df[TARGET_COL]
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]
revenue_test = test_df[REVENUE_COL]

numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print("numeric", len(numeric_cols))
print("categorical", len(categorical_cols), categorical_cols[:10])


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

## 6. Evaluation Functions

For account prioritization, AUC alone is not enough. We care about the top-ranked accounts.

Metrics reported for every model:

- ROC-AUC
- PR-AUC
- Log loss
- Brier score
- Precision@10%
- Recall@10%
- Lift@10%
- Precision@50 accounts
- Recall@50 accounts
- Revenue captured@10%
- Revenue captured@50 accounts

In [ ]:
def precision_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    if k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.mean(np.asarray(y_true)[order]))


def recall_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    positives = float(np.sum(y_true))
    if positives == 0 or k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.sum(np.asarray(y_true)[order]) / positives)


def revenue_capture_at_k(revenue: pd.Series, scores: np.ndarray, k: int) -> float:
    total_revenue = float(np.sum(revenue))
    if total_revenue == 0 or k <= 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    return float(np.sum(np.asarray(revenue)[order]) / total_revenue)


def evaluate_predictions(
    model_name: str,
    y_true: pd.Series,
    scores: np.ndarray,
    revenue: pd.Series,
    top_fraction: float = 0.10,
    top_n: int = 50,
) -> dict[str, Any]:
    y_true_arr = np.asarray(y_true)
    scores = np.clip(np.asarray(scores), 1e-6, 1 - 1e-6)
    k_fraction = max(1, int(len(y_true_arr) * top_fraction))
    k_n = min(top_n, len(y_true_arr))
    base_rate = float(np.mean(y_true_arr))

    return {
        "model": model_name,
        "rows": len(y_true_arr),
        "base_rate": base_rate,
        "roc_auc": roc_auc_score(y_true_arr, scores) if len(np.unique(y_true_arr)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true_arr, scores) if len(np.unique(y_true_arr)) > 1 else np.nan,
        "log_loss": log_loss(y_true_arr, scores, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true_arr, scores),
        "precision_at_10pct": precision_at_k(y_true, scores, k_fraction),
        "recall_at_10pct": recall_at_k(y_true, scores, k_fraction),
        "lift_at_10pct": safe_divide(precision_at_k(y_true, scores, k_fraction), base_rate),
        "revenue_capture_at_10pct": revenue_capture_at_k(revenue, scores, k_fraction),
        "precision_at_50": precision_at_k(y_true, scores, k_n),
        "recall_at_50": recall_at_k(y_true, scores, k_n),
        "lift_at_50": safe_divide(precision_at_k(y_true, scores, k_n), base_rate),
        "revenue_capture_at_50": revenue_capture_at_k(revenue, scores, k_n),
    }


def calibration_table(y_true: pd.Series, scores: np.ndarray, bins: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"y": y_true.values, "score": scores})
    df["score_bin"] = pd.qcut(df["score"], q=bins, duplicates="drop")
    return (
        df.groupby("score_bin", observed=True)
        .agg(
            accounts=("y", "size"),
            avg_predicted_probability=("score", "mean"),
            actual_win_rate=("y", "mean"),
            wins=("y", "sum"),
        )
        .reset_index()
    )

## 7. Probability Calibration

We fit each base model on the training period, then calibrate probabilities on the validation period using Platt scaling.

This avoids using the test period for calibration.

In [ ]:
@dataclass
class FittedModel:
    name: str
    pipeline: Pipeline
    calibrator: LogisticRegression
    validation_metrics: dict[str, Any]
    test_metrics: dict[str, Any]
    valid_scores: np.ndarray
    test_scores: np.ndarray


def logit_transform(probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
    return np.log(probabilities / (1 - probabilities)).reshape(-1, 1)


def fit_calibrated_model(name: str, estimator: Any) -> FittedModel:
    pipeline = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)

    raw_valid = pipeline.predict_proba(X_valid)[:, 1]
    calibrator = LogisticRegression(random_state=RANDOM_STATE)
    calibrator.fit(logit_transform(raw_valid), y_valid)

    valid_scores = calibrator.predict_proba(logit_transform(raw_valid))[:, 1]
    raw_test = pipeline.predict_proba(X_test)[:, 1]
    test_scores = calibrator.predict_proba(logit_transform(raw_test))[:, 1]

    return FittedModel(
        name=name,
        pipeline=pipeline,
        calibrator=calibrator,
        validation_metrics=evaluate_predictions(name, y_valid, valid_scores, valid_df[REVENUE_COL]),
        test_metrics=evaluate_predictions(name, y_test, test_scores, revenue_test),
        valid_scores=valid_scores,
        test_scores=test_scores,
    )

## 8. Train Multiple Models

Models included:

1. Logistic Regression: interpretable baseline
2. Random Forest: nonlinear interactions and robustness
3. Gradient Boosting: strong tabular baseline
4. Optional XGBoost: included if the package is installed

Every model is evaluated using the same metrics.

In [ ]:
models: dict[str, Any] = {
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=20,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.04,
        max_depth=3,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
    ),
}

try:
    from xgboost import XGBClassifier

    models["xgboost"] = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )
except Exception as exc:
    print("XGBoost not available; skipping optional xgboost model.", exc)

fitted_models: list[FittedModel] = []
for name, estimator in models.items():
    print(f"Training {name}...")
    fitted_models.append(fit_calibrated_model(name, estimator))

validation_results = pd.DataFrame([m.validation_metrics for m in fitted_models]).sort_values(
    ["revenue_capture_at_10pct", "pr_auc"], ascending=False
)
test_results = pd.DataFrame([m.test_metrics for m in fitted_models]).sort_values(
    ["revenue_capture_at_10pct", "pr_auc"], ascending=False
)

print("Validation results")
display(validation_results)
print("Test results")
display(test_results)

## 9. Calibration Diagnostics

A calibrated model should have predicted probabilities that roughly match observed win rates.

For example, accounts scored around 0.30 should convert around 30% of the time.

In [ ]:
best_model_name = validation_results.iloc[0]["model"]
best_model = next(model for model in fitted_models if model.name == best_model_name)

print("Best model selected by validation:", best_model.name)

calibration_valid = calibration_table(y_valid, best_model.valid_scores)
calibration_test = calibration_table(y_test, best_model.test_scores)

print("Validation calibration")
display(calibration_valid)
print("Test calibration")
display(calibration_test)

## 10. Precision / Recall Tradeoff

This helps choose an operating threshold if the team wants a fixed score cutoff instead of top-K account routing.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_valid, best_model.valid_scores)
threshold_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})
threshold_table["f1"] = 2 * threshold_table["precision"] * threshold_table["recall"] / (
    threshold_table["precision"] + threshold_table["recall"]
).replace(0, np.nan)
threshold_table = threshold_table.sort_values("f1", ascending=False)
threshold_table.head(20)

## 11. Ranked Account Output

This is the deliverable Growth/Sales can use:

- Account score
- Predicted win probability
- Account metadata
- Actual test outcome for backtesting
- Future 90-day revenue for evaluation

In [ ]:
account_scores = test_df[[
    "snapshot_date",
    "account_id",
    "account_name",
    "industry",
    "segment",
    "estimated_annual_recurring_revenue",
    "target",
    "target_revenue_90d",
]].copy()
account_scores["model"] = best_model.name
account_scores["predicted_win_probability_90d"] = best_model.test_scores
account_scores["priority_rank"] = account_scores["predicted_win_probability_90d"].rank(
    ascending=False, method="first"
).astype(int)
account_scores = account_scores.sort_values("predicted_win_probability_90d", ascending=False)

account_scores.head(30)

## 12. Feature Importance

For tree-based models, use model feature importances when available. For logistic regression, use absolute coefficient values.

For production, I would add SHAP explanations; this notebook keeps dependencies lighter and portable.

In [ ]:
def get_feature_names(pipeline: Pipeline) -> list[str]:
    transformer = pipeline.named_steps["preprocess"]
    try:
        return transformer.get_feature_names_out().tolist()
    except Exception:
        return [f"feature_{i}" for i in range(transformer.transform(X_train.head(1)).shape[1])]


def get_feature_importance(fitted: FittedModel, top_n: int = 40) -> pd.DataFrame:
    model = fitted.pipeline.named_steps["model"]
    feature_names = get_feature_names(fitted.pipeline)

    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_[0])
    else:
        return pd.DataFrame({"message": ["Model does not expose native feature importance."]})

    return (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

feature_importance = get_feature_importance(best_model)
feature_importance

## 13. Save Outputs

These files can feed the app, a BI dashboard, or a GTM workflow.

In [ ]:
OUTPUT_DIR = Path("ml_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

validation_results.to_csv(OUTPUT_DIR / "validation_model_metrics.csv", index=False)
test_results.to_csv(OUTPUT_DIR / "test_model_metrics.csv", index=False)
calibration_valid.to_csv(OUTPUT_DIR / "validation_calibration.csv", index=False)
calibration_test.to_csv(OUTPUT_DIR / "test_calibration.csv", index=False)
account_scores.to_csv(OUTPUT_DIR / "account_priority_scores.csv", index=False)
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

print("Saved outputs to", OUTPUT_DIR.resolve())

## 14. How To Interpret Results

A good senior-level readout should focus on business lift, not only statistical metrics.

Recommended executive summary format:

1. **Top-decile lift:** How much better are the top 10% scored accounts than random?
2. **Revenue capture:** What percent of future won revenue is captured by the top-ranked accounts?
3. **Calibration:** Are predicted probabilities reliable enough for routing decisions?
4. **Drivers:** Which product behaviors and GTM signals explain high scores?
5. **Action:** Which accounts should Sales prioritize this week and why?

Production next steps:

- Replace deal `created_date` target with true close/stage transition date
- Add SHAP explanations
- Backtest by quarter
- Monitor data drift and calibration drift
- Retrain monthly
- Expose scores in the Product Usage Analytics page or CRM